In [1]:
# Cell 1 — Imports and paths
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import torch
import torchvision.models as models
import torchvision.transforms as transforms

MANIFEST_PATH = Path("/kaggle/input/datasets/mohammedumairuddin15/medical-label-auditor-subset/master_manifest.csv")
CHEXPERT_ROOT = Path("/kaggle/input/datasets/ashery/chexpert")
ISIC_ROOT     = Path("/kaggle/input/datasets/nischaydnk/isic-2019-jpg-224x224-resized")
OUTPUT_ROOT   = Path("/kaggle/working")

print("Manifest exists:", MANIFEST_PATH.exists())
print("CheXpert exists:", CHEXPERT_ROOT.exists())
print("ISIC exists:    ", ISIC_ROOT.exists())

Manifest exists: True
CheXpert exists: True
ISIC exists:     True


In [5]:
# Cell 2 — Load new larger manifests
# Cell 2 — Load manifests and fix dataset column
chex_manifest = pd.read_csv(MANIFEST_PATH.parent /
                             "chexpert_subset_manifest.csv")
isic_manifest = pd.read_csv(MANIFEST_PATH.parent /
                             "isic_subset_manifest.csv")

print("CheXpert manifest shape:", chex_manifest.shape)
print("ISIC manifest shape:", isic_manifest.shape)
print("CheXpert columns:", chex_manifest.columns.tolist())
print("ISIC columns:", isic_manifest.columns.tolist())

manifest = pd.concat(
    [chex_manifest, isic_manifest], ignore_index=True
)

# Fix NaN dataset values using image_id pattern
manifest["dataset"] = manifest.apply(
    lambda row: "isic"
    if str(row["image_id"]).startswith("ISIC")
    else "chexpert",
    axis=1
)

print("\nAfter fix:")
print("Combined manifest shape:", manifest.shape)
print(manifest["dataset"].value_counts())
print("\nSample ISIC row:")
print(manifest[manifest["dataset"] == "isic"].head(1))

CheXpert manifest shape: (8776, 6)
ISIC manifest shape: (10000, 5)
CheXpert columns: ['image_id', 'image_path', 'dataset', 'label', 'is_uncertain_original', 'patient_id']
ISIC columns: ['image_id', 'image_path', 'label', 'is_uncertain_original', 'patient_id']

After fix:
Combined manifest shape: (18776, 6)
dataset
isic        10000
chexpert     8776
Name: count, dtype: int64

Sample ISIC row:
          image_id           image_path dataset label  is_uncertain_original  \
8776  ISIC_0068849  NV/ISIC_0068849.jpg    isic    NV                  False   

        patient_id  
8776  ISIC_0068849  


In [6]:
import os

# Build a lookup dictionary of all available ISIC images for fast matching
isic_image_dir = ISIC_ROOT / "train-image" / "image"
isic_files = os.listdir(isic_image_dir)

# Map base image ID to actual filename
isic_lookup = {}
for fname in isic_files:
    base = fname.replace("_downsampled", "").replace(".jpg", "")
    isic_lookup[base] = fname

print(f"Total ISIC files found: {len(isic_files)}")
print(f"Sample lookup entries: {list(isic_lookup.items())[:5]}")

def remap_path(row):
    if row["dataset"] == "chexpert":
        return str(CHEXPERT_ROOT / row["image_path"])
    else:
        actual_filename = isic_lookup.get(row["image_id"])
        if actual_filename:
            return str(isic_image_dir / actual_filename)
        return None

manifest["kaggle_path"] = manifest.apply(remap_path, axis=1)

# Check how many resolved
missing = manifest[manifest["kaggle_path"].isna()]
print(f"Unresolved ISIC paths: {len(missing)}")

# Verify 3 of each
for dataset in ["chexpert", "isic"]:
    sample = manifest[manifest["dataset"] == dataset].head(3)
    for _, row in sample.iterrows():
        p = Path(row["kaggle_path"])
        print(f"{dataset}: {p.name} → exists: {p.exists()}")

Total ISIC files found: 25331
Sample lookup entries: [('ISIC_0057312', 'ISIC_0057312.jpg'), ('ISIC_0014233', 'ISIC_0014233_downsampled.jpg'), ('ISIC_0059626', 'ISIC_0059626.jpg'), ('ISIC_0056156', 'ISIC_0056156.jpg'), ('ISIC_0030912', 'ISIC_0030912.jpg')]
Unresolved ISIC paths: 842
chexpert: view1_frontal.jpg → exists: True
chexpert: view1_frontal.jpg → exists: True
chexpert: view1_frontal.jpg → exists: True
isic: ISIC_0068849.jpg → exists: True
isic: ISIC_0068458.jpg → exists: True
isic: ISIC_0000011.jpg → exists: True


In [7]:
# Add this after Cell 3 in Kaggle Notebook 06
print("Manifest dataset counts:")
print(manifest["dataset"].value_counts())

print("\nSample ISIC paths:")
isic_sample = manifest[manifest["dataset"] == "isic"].head(3)
for _, row in isic_sample.iterrows():
    p = Path(row["kaggle_path"])
    print(f"  {p} → exists: {p.exists()}")

print("\nSample CheXpert paths:")
chex_sample = manifest[manifest["dataset"] == "chexpert"].head(3)
for _, row in chex_sample.iterrows():
    p = Path(row["kaggle_path"])
    print(f"  {p} → exists: {p.exists()}")

Manifest dataset counts:
dataset
isic        10000
chexpert     8776
Name: count, dtype: int64

Sample ISIC paths:
  /kaggle/input/datasets/nischaydnk/isic-2019-jpg-224x224-resized/train-image/image/ISIC_0068849.jpg → exists: True
  /kaggle/input/datasets/nischaydnk/isic-2019-jpg-224x224-resized/train-image/image/ISIC_0068458.jpg → exists: True
  /kaggle/input/datasets/nischaydnk/isic-2019-jpg-224x224-resized/train-image/image/ISIC_0000011.jpg → exists: True

Sample CheXpert paths:
  /kaggle/input/datasets/ashery/chexpert/train/patient17488/study1/view1_frontal.jpg → exists: True
  /kaggle/input/datasets/ashery/chexpert/train/patient10970/study64/view1_frontal.jpg → exists: True
  /kaggle/input/datasets/ashery/chexpert/train/patient30098/study2/view1_frontal.jpg → exists: True


In [8]:
# Drop unresolved paths
before = len(manifest)
manifest = manifest.dropna(subset=["kaggle_path"]).reset_index(drop=True)
after = len(manifest)
print(f"Dropped {before - after} unresolved images")
print(f"Proceeding with {after} images")
print(manifest["dataset"].value_counts())

Dropped 842 unresolved images
Proceeding with 17934 images
dataset
isic        9158
chexpert    8776
Name: count, dtype: int64


In [9]:
# Cell 4 — Verify all paths resolve before running extraction
missing = manifest[~manifest["kaggle_path"].apply(lambda p: Path(p).exists())]
print(f"Missing files: {len(missing)}")
if len(missing) > 0:
    print(missing[["image_id", "dataset", "kaggle_path"]].head(10))
else:
    print("All paths verified. Safe to proceed.")



Missing files: 0
All paths verified. Safe to proceed.


In [11]:
# Cell 5 — Define preprocessing transform
# ResNet18 expects 224x224, RGB, ImageNet-normalised
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [12]:
# Cell 6 — Load frozen ResNet18 backbone (no classification head)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Remove the final FC classification layer — we want the 512-dim feature vector
backbone.fc = torch.nn.Identity()

# Freeze all parameters — we are not training anything
for param in backbone.parameters():
    param.requires_grad = False

backbone = backbone.to(device)
backbone.eval()

print("Backbone loaded. Output dimension: 512")

Device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 159MB/s] 


Backbone loaded. Output dimension: 512


In [13]:
# Cell 7 — Helper function to load and preprocess one image
def load_image(image_path, dataset):
    img = Image.open(image_path)

    # CheXpert X-rays are grayscale — convert to RGB by replicating channels
    if img.mode != "RGB":
        img = img.convert("RGB")

    return transform(img)

In [14]:
# Cell 8 — Extract embeddings for all images
# Runs in batches for efficiency — batch size 64 works well on T4

BATCH_SIZE = 128
all_embeddings = []
failed_indices = []

paths = manifest["kaggle_path"].tolist()
n = len(paths)

print(f"Extracting embeddings for {n} images...")

with torch.no_grad():
    for start in tqdm(range(0, n, BATCH_SIZE)):
        batch_paths = paths[start:start + BATCH_SIZE]
        batch_datasets = manifest["dataset"].tolist()[start:start + BATCH_SIZE]
        batch_tensors = []

        for i, (p, ds) in enumerate(zip(batch_paths, batch_datasets)):
            try:
                tensor = load_image(p, ds)
                batch_tensors.append(tensor)
            except Exception as e:
                print(f"Failed: {p} — {e}")
                # Use a zero vector as placeholder so indices stay aligned
                batch_tensors.append(torch.zeros(3, 224, 224))
                failed_indices.append(start + i)

        batch = torch.stack(batch_tensors).to(device)
        embeddings = backbone(batch).cpu().numpy()
        all_embeddings.append(embeddings)

embeddings_matrix = np.vstack(all_embeddings)
print("Embeddings shape:", embeddings_matrix.shape)
print("Failed images:", len(failed_indices))

Extracting embeddings for 17934 images...


100%|██████████| 141/141 [02:18<00:00,  1.02it/s]

Embeddings shape: (17934, 512)
Failed images: 0


In [15]:
# Cell 9 — Save embeddings matrix and index

# Save the embeddings array (shape: N x 512)
np.save(OUTPUT_ROOT / "embeddings_resnet18.npy", embeddings_matrix)
print("Saved embeddings_resnet18.npy")

# Save the index — maps each row in the matrix back to image metadata
embedding_index = manifest[["image_id", "dataset", "label", 
                              "is_uncertain_original", "kaggle_path"]].copy()
embedding_index["embedding_row"] = range(len(embedding_index))

# Flag any failed rows
embedding_index["extraction_failed"] = False
for idx in failed_indices:
    embedding_index.loc[idx, "extraction_failed"] = True

embedding_index.to_csv(OUTPUT_ROOT / "embedding_index.csv", index=False)
print("Saved embedding_index.csv")
print(embedding_index.head())

Saved embeddings_resnet18.npy
Saved embedding_index.csv
            image_id   dataset       label  is_uncertain_original  \
0  view1_frontal.jpg  chexpert  No Finding                  False   
1  view1_frontal.jpg  chexpert  No Finding                  False   
2  view1_frontal.jpg  chexpert  No Finding                  False   
3  view1_frontal.jpg  chexpert  No Finding                  False   
4  view1_frontal.jpg  chexpert  No Finding                  False   

                                         kaggle_path  embedding_row  \
0  /kaggle/input/datasets/ashery/chexpert/train/p...              0   
1  /kaggle/input/datasets/ashery/chexpert/train/p...              1   
2  /kaggle/input/datasets/ashery/chexpert/train/p...              2   
3  /kaggle/input/datasets/ashery/chexpert/train/p...              3   
4  /kaggle/input/datasets/ashery/chexpert/train/p...              4   

   extraction_failed  
0              False  
1              False  
2              False  
3         

In [16]:
# Cell 10 — Sanity check
embeddings_loaded = np.load(OUTPUT_ROOT / "embeddings_resnet18.npy")
index_loaded = pd.read_csv(OUTPUT_ROOT / "embedding_index.csv")

print("Embeddings shape:", embeddings_loaded.shape)
print("Index shape:", index_loaded.shape)
print("Failed extractions:", index_loaded["extraction_failed"].sum())
print("Per-dataset counts:")
print(index_loaded["dataset"].value_counts())
print("\nSample embedding (first 10 values of row 0):")
print(embeddings_loaded[0][:10])
print("\nAll checks done. Download these two files:")
print("  /kaggle/working/embeddings_resnet18.npy")
print("  /kaggle/working/embedding_index.csv")

Embeddings shape: (17934, 512)
Index shape: (17934, 7)
Failed extractions: 0
Per-dataset counts:
dataset
isic        9158
chexpert    8776
Name: count, dtype: int64

Sample embedding (first 10 values of row 0):
[0.43371284 1.2095455  0.04623969 3.0826936  0.7004517  0.06853079
 0.50680745 0.2582426  0.30786297 1.4708806 ]

All checks done. Download these two files:
  /kaggle/working/embeddings_resnet18.npy
  /kaggle/working/embedding_index.csv
